# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

import ast
import operator as op

_ALLOWED_OPERATORS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg,
    ast.Mod: op.mod, ast.FloorDiv: op.floordiv,
}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (no arbitrary eval)."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree)
        return str(result)
    except Exception:
        return "Error in calculation"


In [1]:
# 🛠️ TOOL 2: Keyword Extractor

import string

_STOPWORDS = {"there", "which", "about", "would", "could", "should", "their"}


def extract_keywords(text: str) -> list:
    """Extract keywords from text (words longer than 4 chars, punctuation stripped, deduped)."""
    try:
        words = text.split()
        cleaned = [w.strip(string.punctuation).lower() for w in words]
        keywords = list(dict.fromkeys(
            w for w in cleaned if len(w) > 4 and w not in _STOPWORDS
        ))
        return keywords[:5]
    except Exception:
        return []


## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [1]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent")


def agent(query: str):
    """Single-agent router: inspects intent, dispatches to a tool, returns structured JSON."""
    logger.info(f"Received query: {query!r}")

    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received")
        return {"type": "error", "result": "Query must be a non-empty string"}

    query_lower = query.lower()

    try:
        if "calculate" in query_lower:
            expression = query_lower.split("calculate", 1)[1].strip()
            logger.info(f"Routed to Calculator Tool with expression: {expression!r}")
            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}
            return {"type": "calculation", "result": calc_result}

        elif "keywords" in query_lower:
            text = query_lower.split("keywords", 1)[1]
            text = text.replace("from", "", 1).strip() if text.strip().startswith("from") else text.strip()
            logger.info(f"Routed to Keyword Extraction Tool with text: {text!r}")
            keywords = extract_keywords(text if text else query)
            return {"type": "keywords", "result": keywords}

        else:
            logger.info("Routed to General Response module")
            return {"type": "general", "result": f"I can help with: '{query}'. Try asking me to calculate something or extract keywords."}

    except Exception as e:
        logger.error(f"Unhandled error while processing query: {e}")
        return {"type": "error", "result": f"Something went wrong: {e}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [1]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'intelligence', 'transforming', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I can help with: 'What is machine learning?'. Try asking me to calculate something or extract keywords."}
--------------------------------------------------


In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

## 🚀 Bonus: Extra Test Cases

A few more queries to show routing, error handling, and the JSON output shape.

In [1]:
# 🎯 Bonus test cases: error handling + edge cases

bonus_queries = [
    "Please calculate 10 / 0",          # error case
    "calculate (3 + 4) * 2",             # nested expression
    "Give me keywords from The quick brown fox jumps over the lazy dog",
    "",                                   # invalid query
]

for q in bonus_queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Please calculate 10 / 0
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: calculate (3 + 4) * 2
Response: {'type': 'calculation', 'result': '14'}
--------------------------------------------------
Query: Give me keywords from The quick brown fox jumps over the lazy dog
Response: {'type': 'keywords', 'result': ['quick', 'brown', 'jumps']}
--------------------------------------------------
Query: 
Response: {'type': 'error', 'result': 'Query must be a non-empty string'}
--------------------------------------------------
